In [ ]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

# Load API key from .env file
load_dotenv()

# Initialize the Anthropic client
client = Anthropic()
# Use Sonnet 4.5 for web search tasks
model = "claude-sonnet-4-5"

In [ ]:
# Helper functions
from anthropic.types import Message


# Handles both raw strings and API Message objects when appending to history
def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


# Sends conversation to Claude with optional tools
def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


# Extracts only text blocks from a response, ignoring tool_use blocks
def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [ ]:
# Web Search Tool — a built-in Anthropic API tool (server-side, no local implementation needed)
# Claude executes the search automatically and returns results inline
web_search_schema = {
    "type": "web_search_20250305",          # API-provided tool type identifier
    "name": "web_search",
    "max_uses": 5,                           # Limit the number of searches per request
    "allowed_domains": ["nih.gov"],           # Restrict searches to specific domains for safety
}

In [ ]:
# Test: Claude will automatically search nih.gov to answer this health question
# The web_search tool is server-side — no run_tools loop needed
messages = []
add_user_message(
    messages,
    """
    What's the best exercise for gaining leg muscle?
    """,
)
# Claude handles the search internally and includes citations in the response
response = chat(messages, tools=[web_search_schema])
response